<a href="https://colab.research.google.com/github/doralalam/llm-engineering/blob/notes/032_create_meeting_minutes_product.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create meeting minutes from an Audio file

I downloaded some Denver City Council meeting minutes and selected a portion of the meeting for us to transcribe. You can download it here:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing

If you'd rather work with the original data, the HuggingFace dataset is [here](https://huggingface.co/datasets/huuuyeah/meetingbank) and the audio can be downloaded [here](https://huggingface.co/datasets/huuuyeah/MeetingBank_Audio/tree/main).

The goal of this product is to use the Audio to generate meeting minutes, including actions.

For this project, you can either use the Denver meeting minutes, or you can record something of your own!


In [ ]:
! pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
from google.colab import drive
from google.colab import userdata
from huggingface_hub import login
import torch
from IPython.display import display, Markdown
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig

In [ ]:
# connect this colab to google drive

drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/ColabNotebooks/LLMEngineering/resources_for_llm_engineering/denver_extract.mp3"

In [ ]:
# login

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credentail=True)

# open the file

audio_file = open(audio_filename, 'rb')

In [ ]:
# constants

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

## STEP 1: Transcribe Audio Using Open Source Model

In [ ]:
# using huggingface pipeline to extract the text from the audio file

pipe = pipeline(
    'automatic-speech-recognition',
    model = "openai/whisper-medium.en",
    dtype = torch.float16,
    device = 'cuda',
    return_timestamps = True
)

result = pipe(audio_filename)
transcription = result['text']
print(transcription)

In [ ]:
open_source_transcription = transcription

## Option 2: Transcribing Audio Using Frontier Closed Source Model

In [ ]:
from openai import OpenAI

In [ ]:
# sign-in to to OpenAI using secrets in colab

AUDIO_MODEL = "gpt-4o-mini-transcribe"

openai_api_key = userdata.get('openai_api_key')
openai = OpenAI(api_key = openai_api_key)
transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_filename, response_format='text')
print(transcription)
closed_source_transcription = transcription

In [ ]:
display(Markdown(open_source_transcription))
print('\n\n')
display(Markdown(closed_source_transcription))

## STEP 2: Analyze and Report

In [ ]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{open_source_transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


## STEP 3: Quantization

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_quant_type = "nf4"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors='pt').to('cuda')
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map = 'auto', quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

In [ ]:
response = tokenizer.decode(outputs[0])

In [ ]:
display(Markdown(response))